# Anomaly Monitor — Exploration Notebook

This notebook walks through the core components of the `anomaly-monitor` project:
1. Generating synthetic data with injected anomalies
2. Building windows with the `Windower`
3. Running the ensemble detector (statistical + LLM stub)
4. Inspecting the LangGraph response agent's decisions

**Prerequisite**: `pip install -e ".[dev,notebooks]"` from the project root.

In [ ]:
import asyncio
import json
from pathlib import Path
from rich import print

from anomaly_monitor.config import settings
from anomaly_monitor.models import Event, Window
from anomaly_monitor.aggregation.windower import Windower
from anomaly_monitor.detection.ensemble import EnsembleDetector
from anomaly_monitor.detection.statistical import StatisticalDetector
from anomaly_monitor.detection.llm_detector import LLMAnomalyDetector

# Force stub mode (no LLM) for reproducibility
settings.openai_api_key = ''
print(f'[bold]Mode:[/bold] {settings.mode.value}  [bold]LLM enabled:[/bold] {settings.llm_enabled}')

## 1. Generate synthetic data

Use the CLI generator to create 1 hour of data at 5 events/sec with 2% anomaly rate.

In [ ]:
!python -m data.generator --hours 1 --rate 5 --anomaly-rate 0.02 \
    --out data/generated/synthetic_1h.jsonl \
    --labels data/generated/labels_1h.jsonl --seed 42

In [ ]:
# Load and inspect
events = [Event.model_validate_json(line) for line in Path('data/generated/synthetic_1h.jsonl').read_text().splitlines() if line.strip()]
labels = [json.loads(line) for line in Path('data/generated/labels_1h.jsonl').read_text().splitlines() if line.strip()]
print(f'Loaded [bold]{len(events)}[/bold] events, [bold]{len(labels)}[/bold] labeled anomaly bursts')
print('Labels:')
for lbl in labels:
    print(f'  - {lbl["anomaly_id"]}: {lbl["kind"]} ({lbl["start_ts"]:.1f} → {lbl["end_ts"]:.1f})')

## 2. Build windows and run the detector

In [ ]:
async def run_detection():
    windower = Windower(settings=settings)
    await windower.start()
    stat = StatisticalDetector(settings=settings)
    llm = LLMAnomalyDetector(settings=settings)
    detector = EnsembleDetector(settings=settings, statistical=stat, llm=llm)

    current_ids = {}
    scores = []
    for ev in events:
        windows = await windower.add_event(ev)
        for dur_key, win in windows.items():
            prev = current_ids.get(dur_key)
            if prev is not None and prev != win.window_id:
                prev_win = await windower.get_window_by_id(prev)
                if prev_win and prev_win.count > 0:
                    score = await detector.detect(prev_win)
                    scores.append((prev_win, score))
            current_ids[dur_key] = win.window_id
    await windower.aclose()
    return scores

scores = asyncio.run(run_detection())
anomalies = [(w, s) for w, s in scores if s.is_anomaly]
print(f'Scored [bold]{len(scores)}[/bold] windows, [bold red]{len(anomalies)}[/bold red] flagged as anomalies')

In [ ]:
# Show the flagged anomalies
for win, score in anomalies:
    print(f'\n[bold red]{win.window_id}[/bold red] (count={win.count}, err_rate={win.error_rate:.3f})')
    print(f'  severity: {score.severity.value}, prob: {score.probability:.3f}')
    print(f'  reason: {score.reason}')

## 3. Run the full eval

Use the CLI to run the full eval with the rubric.

In [ ]:
!python -m eval.run_eval --data data/generated/synthetic_1h.jsonl \
    --labels data/generated/labels_1h.jsonl --no-judge --speed 10000